# 自定义实体识别器训练与推理（ModelArts + BERT-NER）

本 Notebook 复刻「从设备名列表训练自定义实体识别器」的完整流程，运行在华为云 ModelArts Notebook（Python 3 + GPU/CPU 均可）中：

1. 准备数据：实体列表 CSV（Text,Type）+ 语料
2. 转换数据：CSV 标注 → CoNLL 格式（供 BERT-NER 使用）
3. 微调模型：使用 `transformers` 加载预训练 BERT 并微调 token 分类
4. 推理验证：对测试文本抽取 DEVICE 实体

> 前置条件：已完成 `terraform apply`（创建 OBS 桶与 ModelArts 委托），
> 并已通过环境变量注入访问密钥（`HW_ACCESS_KEY` / `HW_SECRET_KEY`）。


## 1. 依赖安装与库导入

In [ ]:
# 安装依赖（如已安装可跳过）
# !pip install transformers datasets tokenizers torch pandas

import os
import json
import time
import pandas as pd

print("环境就绪")

## 2. 数据准备

In [ ]:
# 实体列表（电信行业设备名称）
# 业务场景：电信运营商需要监控用户在社交媒体上提及的手机品牌/型号，
# 以支持设备市场份额分析、竞品监控、终端故障关联分析等业务场景
#
# 实体类型说明：
# - DEVICE: 手机、平板等终端设备
# - 扩展建议：可添加 "5G", "4G", "LTE", "SIM卡" 等网络制式实体

# 设备名称列表（涵盖主流智能手机品牌：Apple、Samsung、国产头部品牌）
phones = [
    # Apple iPhone 系列
    'iPhone 15', 'iPhone 15 Pro', 'iPhone 15 Pro Max',
    'iPhone 14', 'iPhone 14 Pro', 'iPhone 14 Pro Max',
    'iPhone 13', 'iPhone 13 Pro', 'iPhone SE',
    'iPhone X', 'iPhone XR', 'iPhone XS', 'iPhone 11', 'iPhone 12',
    'iPhone', 'iPad', 'iPad Pro', 'Apple Watch',
    # Samsung Galaxy 系列
    'Samsung Galaxy', 'Galaxy S24', 'Galaxy S23', 'Galaxy S22', 'Galaxy S21',
    'Galaxy Note', 'Galaxy Z Fold', 'Galaxy Z Flip', 'Galaxy A', 'Galaxy M',
    'Samsung', 'Samsung Galaxy', 'Galaxy',
    # 国产品牌（华为、小米、OPPO、vivo、荣耀等）
    'Huawei', 'Huawei P70', 'Huawei P60', 'Huawei Mate 60', 'Huawei Mate 50',
    'Xiaomi', 'Xiaomi 14', 'Xiaomi 13', 'Redmi', 'Mi',
    'OPPO', 'OPPO Find X', 'OPPO Reno', 'OnePlus',
    'vivo', 'vivo X', 'vivo S', 'iQOO',
    'Honor', 'Honor Magic', 'Honor 90', 'Honor 70',
    # 其他品牌
    'Google Pixel', 'Pixel', 'Motorola', 'Nokia', 'Sony Xperia',
    # 操作系统/生态
    'Android', 'android', 'iOS', 'ios', '鸿蒙', 'HarmonyOS',
    # 泛指
    'smartphone', 'mobile phone', 'cellphone', '手机'
]

# 去重并保存
phones = list(set(phones))
df_entity_list = pd.DataFrame(phones, columns=['Text'])
df_entity_list['Type'] = 'DEVICE'
df_entity_list.to_csv('entity_list.csv', encoding='utf-8', index=False)
print(f"实体列表已生成，共 {len(df_entity_list)} 个设备名称")
print(df_entity_list.head(10))

In [ ]:
# 语料：读取推文文本（每行一句）
tweet_telco = pd.read_csv('../../data/tweet_telco.csv', encoding='utf-8', header=None, names=['text'])
tweet_telco['text'].to_csv('telco_corpus.txt', encoding='utf-8', index=False, header=False)
print(f"语料行数: {len(tweet_telco)}")

## 3. CSV 标注 → CoNLL 格式

In [ ]:
# 调用仓库内置转换脚本（或直接使用下方等价代码）
import subprocess, sys
subprocess.run([
    sys.executable, '../scripts/csv_to_conll.py',
    '--entity-list', 'entity_list.csv',
    '--corpus', 'telco_corpus.txt',
    '--output', 'train.conll',
    '--entity-type', 'DEVICE',
], check=True)

## 4. 微调 BERT-NER（transformers TokenClassification）

In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
)

# 预训练模型（ModelArts 上可从 HF Hub 拉取；也可替换为本地/昇腾适配模型）
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 读取 CoNLL 数据
def read_conll(path):
    sentences, labels = [], []
    sent, lab = [], []
    for line in open(path, encoding='utf-8'):
        line = line.strip()
        if not line:
            if sent:
                sentences.append(sent); labels.append(lab)
                sent, lab = [], []
            continue
        t, l = line.split('\t')
        sent.append(t); lab.append(l)
    if sent:
        sentences.append(sent); labels.append(lab)
    return sentences, labels

sentences, labels = read_conll('train.conll')
print(f"句子数: {len(sentences)}, 样例: {sentences[0][:8]}")

In [ ]:
# 构造标签集合与 id 映射
all_labels = sorted({l for labs in labels for l in labs})
label2id = {l: i for i, l in enumerate(all_labels)}
id2label = {i: l for l, i in label2id.items()}
print("标签集合:", all_labels)

In [ ]:
# tokenize + 对齐标签（word_ids 对齐）
def align_labels(words, tags):
    enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=128)
    word_ids = enc.word_ids()
    label_ids = []
    prev = None
    for wid in word_ids:
        if wid is None:
            label_ids.append(-100)
        elif wid != prev:
            label_ids.append(label2id[tags[wid]])
        else:
            label_ids.append(label2id[tags[wid]])
        prev = wid
    enc["labels"] = label_ids
    return {k: torch.tensor(v) for k, v in enc.items()}

dataset = [align_labels(s, l) for s, l in zip(sentences, labels)]
print(f"样本数: {len(dataset)}")

In [ ]:
# 微调配置（CPU 环境小样本足够演示；GPU 可增大 epoch）
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(all_labels),
    id2label=id2label, label2id=label2id,
)

training_args = TrainingArguments(
    output_dir='./ner_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    save_total_limit=1,
    logging_steps=50,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
)
trainer.train()
print("训练完成")

## 5. 推理验证（抽取 DEVICE 实体）

In [ ]:
def predict_entities(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**tokens).logits
    preds = logits.argmax(-1)[0].tolist()
    words = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])
    out = []
    for w, p in zip(words, preds):
        label = id2label[p]
        if label != 'O' and not w.startswith('##'):
            out.append((w, label))
    return out

test = "I just bought a new iPhone X and my Samsung Galaxy arrived yesterday, but the Android one is still on backorder."
for w, l in predict_entities(test):
    print(f"{w:20s} -> {l}")

## 6. 保存模型与产物（可选：上传 OBS）

In [ ]:
# 保存模型
model.save_pretrained('./ner_model')
tokenizer.save_pretrained('./ner_model')
print("模型已保存到 ./ner_model")

# 可选：上传到 OBS（示例，使用 obsutil）
# !obsutil cp -r ./ner_model obs://<bucket>/custom-nlp-pipeline/model/ner_model